# AI Scam Shield v2 — Full Multi-Source Training & 100-Sample Benchmark

Notebook huấn luyện v2 cho **AI Scam Shield** trên toàn bộ 35 shard `Data_Preprocessing.zip` (`bangta205/scam-detection-data-preprocessing-full`), kết hợp:
- `ngovanhh/vi-fraud-bench-vi` (lừa đảo & hội thoại sạch tiếng Việt)
- `tuan251209/vietnamese-sms-dataset` (tin nhắn OTP ngân hàng sạch vs SMS lừa đảo tại Việt Nam)
- `naserabdullahalam/phishing-email-dataset` (email lừa đảo tài chính / mạo danh / công việc vs email công việc hợp lệ)
- `uciml/sms-spam-collection-dataset` (SMS tiếng Anh)

**Cải tiến v2 khắc phục các lỗi ở v1:**
1. Loại bỏ double-upsampling (`WeightedRandomSampler` + nhân 3x cùng lúc) gây lệch xác suất prior ở tiếng Việt và làm ngưỡng tối ưu bị kẹp sát `0.85`.
2. Cân bằng tỷ lệ dữ liệu Telegram cấp kênh (~95k Clean phủ đủ 35 shard + 100% Scam duy nhất ~55k) với dữ liệu cấp tin nhắn/email thực tế để tránh kênh tin tức crypto làm nhiễu mẫu phishing.
3. Bảo toàn token `[URL]` kèm tên miền đã chuẩn hóa (`domain host`) thay vì xóa sạch hoàn toàn domain lừa đảo như `csgt-tracuuphat.com`, `binance-kyc-fix.com`.


In [ ]:
import os, gc, glob, json, time, math, hashlib, re, zipfile
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, classification_report
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpu = torch.cuda.device_count()
print(f"Device: {device} | GPUs: {n_gpu}")
if torch.cuda.is_available():
    for i in range(n_gpu):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
CONFIG = {
    "model_name": "xlm-roberta-base",
    "max_length": 128,
    "batch_size": 128,
    "epochs": 3,
    "lr": 2.5e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "label_smoothing": 0.02,
    "clean_per_shard_cap": 2800,
    "vi_clean_extra_cap": 20000,
    "phish_per_class_cap": 25000,
    "output_dir": "/kaggle/working/exported_model",
    "artifacts_dir": "/kaggle/working/artifacts",
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["artifacts_dir"], exist_ok=True)


def find_input_dir(slug: str) -> str:
    for p in glob.glob(f"/kaggle/input/**/{slug}", recursive=True):
        if os.path.isdir(p):
            return p
    direct = f"/kaggle/input/{slug}"
    if os.path.isdir(direct):
        return direct
    raise FileNotFoundError(f"Dataset not found for slug: {slug}")


TG_DIR = find_input_dir("scam-detection-data-preprocessing-full")
VI_DIR = find_input_dir("vi-fraud-bench-vi")
SMS_DIR = find_input_dir("sms-spam-collection-dataset")
VI_SMS_DIR = find_input_dir("vietnamese-sms-dataset")
PHISH_DIR = find_input_dir("phishing-email-dataset")
print("TG_DIR    :", TG_DIR)
print("VI_DIR    :", VI_DIR)
print("SMS_DIR   :", SMS_DIR)
print("VI_SMS_DIR:", VI_SMS_DIR)
print("PHISH_DIR :", PHISH_DIR)

RE_URL = re.compile(
    r"(?:https?://|www\.|t\.me/|bit\.ly/|zalo\.me/)([^\s\]\[\"'<>()]+)|"
    r"\b([a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.(?:com|vn|net|org|xyz|top|vip|club|info|io|co|me|cc|tk|ml|ga|cf))(?:/[^\s]*)?",
    re.IGNORECASE,
)
RE_UNDERSCORE_JOIN = re.compile(r"(?<=[0-9A-Za-z\u00C0-\u1EF9])_(?=[0-9A-Za-z\u00C0-\u1EF9])")
RE_WHITESPACE = re.compile(r"\s+")
RE_VI_CHAR = re.compile(r"[àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]", re.IGNORECASE)


def _replace_url_with_domain(match: re.Match) -> str:
    raw = match.group(0).lower()
    raw = re.sub(r"^https?://", "", raw)
    host = raw.split("/")[0].split("?")[0]
    host_words = re.sub(r"[^a-z0-9\-]+", " ", host).replace("-", " ").strip()
    return f" {host_words} " if host_words else " "


def normalize_text(text: str, has_url_hint: bool | None = None) -> str:
    if not isinstance(text, str):
        text = "" if text is None else str(text)
    text = text.replace("\r", " ").replace("\n", " ")
    detected_url = bool(RE_URL.search(text))
    text = RE_URL.sub(_replace_url_with_domain, text)
    text = RE_UNDERSCORE_JOIN.sub(" ", text)
    text = RE_WHITESPACE.sub(" ", text).strip().lower()
    if not text:
        return ""
    has_url = detected_url if has_url_hint is None else (bool(has_url_hint) or detected_url)
    if has_url:
        return f"[url] {text}"
    return text


def text_hash(s: str) -> int:
    return int.from_bytes(hashlib.blake2b(s.encode("utf-8", errors="ignore"), digest_size=8).digest(), "little")

In [ ]:
parquet_files = sorted(glob.glob(os.path.join(TG_DIR, "*.parquet")))
print(f"Found {len(parquet_files)} parquet shards in TG_DIR")

seen_hashes = set()
tg_scam_rows = []
tg_clean_rows = []
tg_vi_clean_rows = []

stats = {
    "raw_total": 0,
    "raw_scam": 0,
    "raw_clean": 0,
    "dropped_null_or_empty": 0,
    "dropped_boilerplate": 0,
    "dropped_dup_scam": 0,
    "kept_unique_scam": 0,
    "kept_clean_sampled": 0,
    "kept_vi_clean": 0,
}

for idx, pf_path in enumerate(parquet_files, 1):
    df = pd.read_parquet(pf_path, columns=["cleaned_text", "has_url", "label"])
    stats["raw_total"] += len(df)

    null_mask = df["cleaned_text"].isna() | df["label"].isna()
    stats["dropped_null_or_empty"] += int(null_mask.sum())
    df = df[~null_mask].copy()

    bp_mask = df["cleaned_text"].str.contains("unavailable due to copyright", case=False, na=False)
    stats["dropped_boilerplate"] += int(bp_mask.sum())
    df = df[~bp_mask]

    df["label_int"] = df["label"].astype(int)
    scam_df = df[df["label_int"] == 1]
    clean_df = df[df["label_int"] == 0]
    stats["raw_scam"] += len(scam_df)
    stats["raw_clean"] += len(clean_df)

    for txt, hurl in zip(scam_df["cleaned_text"].tolist(), scam_df["has_url"].tolist()):
        norm = normalize_text(txt, bool(hurl))
        if len(norm) < 8:
            stats["dropped_null_or_empty"] += 1
            continue
        h = text_hash(norm)
        if h in seen_hashes:
            stats["dropped_dup_scam"] += 1
            continue
        seen_hashes.add(h)
        tg_scam_rows.append((norm, 1, "tg_data"))
        stats["kept_unique_scam"] += 1

    if len(clean_df) > 0:
        if stats["kept_vi_clean"] < CONFIG["vi_clean_extra_cap"]:
            vi_mask = clean_df["cleaned_text"].str.contains(RE_VI_CHAR, na=False)
            vi_sub = clean_df[vi_mask]
            if len(vi_sub) > 0:
                take_n = min(len(vi_sub), 1000, CONFIG["vi_clean_extra_cap"] - stats["kept_vi_clean"])
                vi_sample = vi_sub.sample(n=take_n, random_state=SEED + idx)
                for txt, hurl in zip(vi_sample["cleaned_text"].tolist(), vi_sample["has_url"].tolist()):
                    norm = normalize_text(txt, bool(hurl))
                    if len(norm) < 8:
                        continue
                    h = text_hash(norm)
                    if h not in seen_hashes:
                        seen_hashes.add(h)
                        tg_vi_clean_rows.append((norm, 0, "tg_vi_clean"))
                        stats["kept_vi_clean"] += 1

        cap = min(CONFIG["clean_per_shard_cap"], len(clean_df))
        cl_sample = clean_df.sample(n=cap, random_state=SEED + idx)
        for txt, hurl in zip(cl_sample["cleaned_text"].tolist(), cl_sample["has_url"].tolist()):
            norm = normalize_text(txt, bool(hurl))
            if len(norm) < 8:
                continue
            h = text_hash(norm)
            if h not in seen_hashes:
                seen_hashes.add(h)
                tg_clean_rows.append((norm, 0, "tg_data"))
                stats["kept_clean_sampled"] += 1

    if idx % 7 == 0 or idx == len(parquet_files):
        print(
            f"[{idx:02d}/{len(parquet_files)}] raw={stats['raw_total']:,} | "
            f"unique_scam={stats['kept_unique_scam']:,} | clean={stats['kept_clean_sampled']:,} | "
            f"vi_clean={stats['kept_vi_clean']:,}"
        )
    del df, scam_df, clean_df
    gc.collect()

print("\nTGDataset Ingestion Summary:", json.dumps(stats, indent=2))

In [ ]:
def load_jsonl_rows(path: str, source_name: str):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            norm = normalize_text(obj.get("text", ""), has_url_hint=None)
            if len(norm) >= 6:
                rows.append((norm, int(obj["y"]), source_name))
    return rows


vi_pool_rows = load_jsonl_rows(os.path.join(VI_DIR, "vi_fraud_pool.jsonl"), "vi_fraud")
vi_val_rows = load_jsonl_rows(os.path.join(VI_DIR, "val.jsonl"), "vi_fraud_val")
vi_test_rows = load_jsonl_rows(os.path.join(VI_DIR, "test.jsonl"), "vi_fraud_test")

# Vietnamese SMS dataset (legit OTP/announcements vs Vietnamese phishing SMS)
visms_tr_df = pd.read_csv(os.path.join(VI_SMS_DIR, "train.csv"))[["message", "label"]].dropna()
visms_te_df = pd.read_csv(os.path.join(VI_SMS_DIR, "test.csv"))[["message", "label"]].dropna()
visms_tr_rows = [(normalize_text(m), int(y), "vi_sms") for m, y in zip(visms_tr_df["message"], visms_tr_df["label"]) if len(normalize_text(m)) >= 6]
visms_te_rows = [(normalize_text(m), int(y), "vi_sms_test") for m, y in zip(visms_te_df["message"], visms_te_df["label"]) if len(normalize_text(m)) >= 6]

# English SMS spam
sms_df = pd.read_csv(os.path.join(SMS_DIR, "spam.csv"), encoding="latin-1")[["v1", "v2"]].dropna()
sms_rows = []
for lbl_str, txt in zip(sms_df["v1"], sms_df["v2"]):
    norm = normalize_text(txt, has_url_hint=None)
    if len(norm) >= 6:
        sms_rows.append((norm, 1 if str(lbl_str).strip().lower() == "spam" else 0, "en_sms"))

# English Phishing vs Legitimate Emails (sample evenly per class, slice first 1000 chars for speed)
phish_df = pd.read_csv(os.path.join(PHISH_DIR, "phishing_email.csv")).dropna(subset=["text_combined", "label"])
phish_pos = phish_df[phish_df["label"].astype(int) == 1].sample(n=min(CONFIG["phish_per_class_cap"], int((phish_df["label"] == 1).sum())), random_state=SEED)
phish_neg = phish_df[phish_df["label"].astype(int) == 0].sample(n=min(CONFIG["phish_per_class_cap"], int((phish_df["label"] == 0).sum())), random_state=SEED)
phish_sub = pd.concat([phish_pos, phish_neg], ignore_index=True)
phish_rows = []
for txt, lbl in zip(phish_sub["text_combined"].tolist(), phish_sub["label"].astype(int).tolist()):
    norm = normalize_text(str(txt)[:1000], has_url_hint=None)
    if len(norm) >= 10:
        phish_rows.append((norm, lbl, "en_phish"))
del phish_df, phish_pos, phish_neg, phish_sub
gc.collect()

# Prevent data leakage across held-out eval sets
reserved_hashes = {text_hash(r[0]) for r in (vi_val_rows + vi_test_rows + visms_te_rows)}

def dedup_list(rows, skip_hashes=None):
    local_seen = set(skip_hashes) if skip_hashes else set()
    out = []
    for text, y, src in rows:
        h = text_hash(text)
        if h in local_seen:
            continue
        local_seen.add(h)
        out.append((text, y, src))
    return out

tg_all = [r for r in (tg_scam_rows + tg_clean_rows + tg_vi_clean_rows) if text_hash(r[0]) not in reserved_hashes]
vi_pool_clean = dedup_list(vi_pool_rows, reserved_hashes)
visms_tr_clean = dedup_list(visms_tr_rows, reserved_hashes)
sms_clean = dedup_list(sms_rows, reserved_hashes)
phish_clean = dedup_list(phish_rows, reserved_hashes)

df_tg = pd.DataFrame(tg_all, columns=["text", "label", "source"])
df_vi = pd.DataFrame(vi_pool_clean, columns=["text", "label", "source"])
df_visms = pd.DataFrame(visms_tr_clean, columns=["text", "label", "source"])
df_sms = pd.DataFrame(sms_clean, columns=["text", "label", "source"])
df_phish = pd.DataFrame(phish_clean, columns=["text", "label", "source"])

def split_90_5_5(df_in):
    tr, tmp = train_test_split(df_in, test_size=0.10, stratify=df_in["label"], random_state=SEED)
    va, te = train_test_split(tmp, test_size=0.50, stratify=tmp["label"], random_state=SEED)
    return tr, va, te

tg_tr, tg_va, tg_te = split_90_5_5(df_tg)
vi_tr, vi_va, vi_te = split_90_5_5(df_vi)
visms_tr, visms_va, visms_te_split = split_90_5_5(df_visms)
sms_tr, sms_va, sms_te = split_90_5_5(df_sms)
ph_tr, ph_va, ph_te = split_90_5_5(df_phish)

# Balance training domains naturally (no WeightedRandomSampler prior shift!)
train_df = pd.concat(
    [tg_tr, vi_tr, vi_tr, visms_tr, visms_tr, visms_tr, visms_tr, sms_tr, sms_tr, ph_tr],
    ignore_index=True,
).sample(frac=1.0, random_state=SEED)

val_df = pd.concat([tg_va, vi_va, visms_va, sms_va, ph_va], ignore_index=True)
test_df = pd.concat([tg_te, vi_te, visms_te_split, sms_te, ph_te], ignore_index=True)
vi_bench_df = pd.DataFrame(vi_test_rows, columns=["text", "label", "source"])
vi_sms_bench_df = pd.DataFrame(visms_te_rows, columns=["text", "label", "source"])

print(f"Train size   : {len(train_df):,} (Scam: {train_df['label'].mean()*100:.2f}%)")
print(f"Val size     : {len(val_df):,} (Scam: {val_df['label'].mean()*100:.2f}%)")
print(f"Test size    : {len(test_df):,} (Scam: {test_df['label'].mean()*100:.2f}%)")
print(f"VI Bench     : {len(vi_bench_df):,} (Scam: {vi_bench_df['label'].mean()*100:.2f}%)")
print(f"VI SMS Bench : {len(vi_sms_bench_df):,} (Scam: {vi_sms_bench_df['label'].mean()*100:.2f}%)")

del tg_scam_rows, tg_clean_rows, tg_vi_clean_rows, tg_all, df_tg, df_vi, df_visms, df_sms, df_phish
gc.collect()

In [ ]:
t0 = time.time()
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=120_000, sublinear_tf=True, min_df=3)
X_tr = tfidf.fit_transform(train_df["text"])
X_te = tfidf.transform(test_df["text"])
X_vi = tfidf.transform(vi_bench_df["text"])

lr_clf = LogisticRegression(max_iter=300, class_weight="balanced", C=2.0, n_jobs=-1)
lr_clf.fit(X_tr, train_df["label"].values)

def eval_probs(y_true, probs, thresh=0.5, name="Model"):
    preds = (probs >= thresh).astype(int)
    return {
        "name": name,
        "accuracy": round(float(accuracy_score(y_true, preds)), 4),
        "precision": round(float(precision_score(y_true, preds, zero_division=0)), 4),
        "recall": round(float(recall_score(y_true, preds, zero_division=0)), 4),
        "f1": round(float(f1_score(y_true, preds, zero_division=0)), 4),
        "roc_auc": round(float(roc_auc_score(y_true, probs)), 4),
        "pr_auc": round(float(average_precision_score(y_true, probs)), 4),
    }

lr_te_probs = lr_clf.predict_proba(X_te)[:, 1]
lr_vi_probs = lr_clf.predict_proba(X_vi)[:, 1]

baseline_test_metrics = eval_probs(test_df["label"].values, lr_te_probs, 0.5, "TF-IDF + LogReg (Combined Test)")
baseline_vi_metrics = eval_probs(vi_bench_df["label"].values, lr_vi_probs, 0.5, "TF-IDF + LogReg (VI Bench)")
print("Baseline Test:", baseline_test_metrics)
print("Baseline VI  :", baseline_vi_metrics)
print(f"Baseline time: {time.time() - t0:.1f}s")

del X_tr, X_te, X_vi, tfidf, lr_clf
gc.collect()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])


class TextScamDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], int(self.labels[idx])


def collate_fn(batch):
    texts, labels = zip(*batch)
    enc = tokenizer(
        list(texts),
        max_length=CONFIG["max_length"],
        padding=True,
        truncation=True,
        return_tensors="pt",
    )
    enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return enc


train_ds = TextScamDataset(train_df["text"].values, train_df["label"].values)
val_ds = TextScamDataset(val_df["text"].values, val_df["label"].values)
test_ds = TextScamDataset(test_df["text"].values, test_df["label"].values)
vi_bench_ds = TextScamDataset(vi_bench_df["text"].values, vi_bench_df["label"].values)
vi_sms_ds = TextScamDataset(vi_sms_bench_df["text"].values, vi_sms_bench_df["label"].values)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"] * 2, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"] * 2, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)
vi_loader = DataLoader(vi_bench_ds, batch_size=CONFIG["batch_size"] * 2, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)
vi_sms_loader = DataLoader(vi_sms_ds, batch_size=CONFIG["batch_size"] * 2, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

model = AutoModelForSequenceClassification.from_pretrained(CONFIG["model_name"], num_labels=2).to(device)
if n_gpu > 1:
    model = nn.DataParallel(model)

no_decay = {"bias", "LayerNorm.weight"}
optimizer_grouped = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": CONFIG["weight_decay"]},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(optimizer_grouped, lr=CONFIG["lr"])
total_steps = len(train_loader) * CONFIG["epochs"]
warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
scaler = torch.amp.GradScaler("cuda")

print(f"Steps/epoch: {len(train_loader):,} | Total steps: {total_steps:,} | Warmup: {warmup_steps:,}")

In [ ]:
@torch.no_grad()
def predict_loader(loader):
    model.eval()
    probs, labels = [], []
    total_loss = 0.0
    for batch in loader:
        ids = batch["input_ids"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        lbl = batch["labels"].to(device, non_blocking=True)
        with torch.amp.autocast("cuda"):
            out = model(input_ids=ids, attention_mask=mask)
            logits = out.logits
            loss = criterion(logits, lbl)
        total_loss += loss.item()
        probs.extend(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().tolist())
        labels.extend(lbl.cpu().tolist())
    return np.array(probs), np.array(labels), total_loss / max(len(loader), 1)


def find_best_threshold(y_true, probs):
    best_f1, best_t = 0.0, 0.5
    for t in np.linspace(0.25, 0.75, 51):
        preds = (probs >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t, best_f1


history = []
best_val_f1 = -1.0
eval_every = max(len(train_loader) // 2, 200)
global_step = 0
t_start = time.time()

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    running_loss = 0.0
    steps_since_eval = 0
    for step, batch in enumerate(train_loader, 1):
        ids = batch["input_ids"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        lbl = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            out = model(input_ids=ids, attention_mask=mask)
            loss = criterion(out.logits, lbl)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()
        steps_since_eval += 1
        global_step += 1

        if step % eval_every == 0 or step == len(train_loader):
            val_probs, val_labels, val_loss = predict_loader(val_loader)
            t_opt, val_f1 = find_best_threshold(val_labels, val_probs)
            val_roc = roc_auc_score(val_labels, val_probs)
            avg_tr_loss = running_loss / max(steps_since_eval, 1)
            running_loss = 0.0
            steps_since_eval = 0

            rec = {
                "epoch": epoch,
                "step": global_step,
                "train_loss": round(avg_tr_loss, 4),
                "val_loss": round(val_loss, 4),
                "val_f1": round(val_f1, 4),
                "val_roc_auc": round(val_roc, 4),
                "best_thresh": round(t_opt, 3),
                "elapsed_min": round((time.time() - t_start) / 60, 2),
            }
            history.append(rec)
            print(f"[Ep {epoch} | Step {global_step:,}] tr_loss={rec['train_loss']:.4f} | val_loss={rec['val_loss']:.4f} | val_f1={rec['val_f1']:.4f} | roc_auc={rec['val_roc_auc']:.4f} | thresh={rec['best_thresh']} | {rec['elapsed_min']}m")

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                base_model = model.module if hasattr(model, "module") else model
                base_model.save_pretrained(CONFIG["output_dir"], safe_serialization=True)
                tokenizer.save_pretrained(CONFIG["output_dir"])
                with open(os.path.join(CONFIG["output_dir"], "threshold.json"), "w") as f:
                    json.dump({"threshold": round(t_opt, 4)}, f)
            model.train()

print(f"Training completed in {(time.time() - t_start)/60:.1f} min | Best Val F1: {best_val_f1:.4f}")

In [ ]:
eval_model = AutoModelForSequenceClassification.from_pretrained(CONFIG["output_dir"]).to(device)
eval_model.eval()
with open(os.path.join(CONFIG["output_dir"], "threshold.json")) as f:
    BEST_THRESHOLD = float(json.load(f)["threshold"])

model = eval_model

t_inf = time.time()
te_probs, te_labels, _ = predict_loader(test_loader)
latency_ms = (time.time() - t_inf) * 1000.0 / max(len(te_labels), 1)
vi_probs, vi_labels, _ = predict_loader(vi_loader)
visms_probs, visms_labels, _ = predict_loader(vi_sms_loader)

xlmr_test_metrics = eval_probs(te_labels, te_probs, BEST_THRESHOLD, "XLM-RoBERTa (Combined Test)")
xlmr_vi_metrics = eval_probs(vi_labels, vi_probs, BEST_THRESHOLD, "XLM-RoBERTa (VI Bench)")
xlmr_visms_metrics = eval_probs(visms_labels, visms_probs, BEST_THRESHOLD, "XLM-RoBERTa (VI SMS Test)")

print("\n=== MODEL COMPARISON ===")
comp_df = pd.DataFrame([baseline_test_metrics, xlmr_test_metrics, baseline_vi_metrics, xlmr_vi_metrics, xlmr_visms_metrics])
print(comp_df.to_string(index=False))
print(f"\nCalibrated threshold: {BEST_THRESHOLD:.4f} | Batch GPU latency/sample: {latency_ms:.3f} ms")

print("\n--- Detailed Classification Report (Combined Test) ---")
print(classification_report(te_labels, (te_probs >= BEST_THRESHOLD).astype(int), target_names=["Safe", "Scam"], digits=4))
print("--- Detailed Classification Report (VI Fraud Bench Test) ---")
print(classification_report(vi_labels, (vi_probs >= BEST_THRESHOLD).astype(int), target_names=["Safe", "Scam"], digits=4))
print("--- Detailed Classification Report (VI SMS Test) ---")
print(classification_report(visms_labels, (visms_probs >= BEST_THRESHOLD).astype(int), target_names=["Safe", "Scam"], digits=4))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (yt, yp, title) in zip(
    axes,
    [
        (te_labels, (te_probs >= BEST_THRESHOLD).astype(int), "Combined Test Set"),
        (vi_labels, (vi_probs >= BEST_THRESHOLD).astype(int), "VI Fraud Bench"),
        (visms_labels, (visms_probs >= BEST_THRESHOLD).astype(int), "VI SMS Test"),
    ],
):
    cm = confusion_matrix(yt, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Safe", "Scam"], yticklabels=["Safe", "Scam"], ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["artifacts_dir"], "confusion_matrix.png"), dpi=150)
plt.show()

if history:
    hdf = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hdf["step"], hdf["train_loss"], label="Train Loss")
    axes[0].plot(hdf["step"], hdf["val_loss"], label="Val Loss")
    axes[0].set_title("Loss Curve"); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(hdf["step"], hdf["val_f1"], marker="o", label="Val F1")
    axes[1].plot(hdf["step"], hdf["val_roc_auc"], marker="s", label="Val ROC-AUC")
    axes[1].set_title("Validation F1 & ROC-AUC"); axes[1].legend(); axes[1].grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["artifacts_dir"], "training_curves.png"), dpi=150)
    plt.show()

In [ ]:
TEST_100_SAMPLES = [
    # ── 25 Vietnamese Scam ──
    ("Chúc mừng bạn đã trúng thưởng 100 triệu đồng từ chương trình tri ân! Truy cập ngay http://qua-tang-vip.xyz để nhận giải.", 1),
    ("Tài khoản ngân hàng của bạn có dấu hiệu bị khóa, vui lòng ấn vào link http://vcb-xacthuc.top để cập nhật bảo mật.", 1),
    ("Cảnh báo từ Cục Cảnh sát: Số CCCD của bạn liên quan đến vụ án rửa tiền, liên hệ ngay để xác minh tài khoản định danh VNeID.", 1),
    ("Tuyển cộng tác viên làm việc tại nhà, chỉ cần like video TikTok nhận ngay 500k mỗi ngày, không cần kinh nghiệm.", 1),
    ("Đầu tư tiền ảo sinh lời 200%/tháng, cam kết hoàn vốn 100%, inbox Telegram để vào nhóm VIP kéo lệnh.", 1),
    ("Cơ hội nhận quà tri ân miễn phí từ điện máy xanh, bấm vào đây nhận ngay phiếu mua hàng 5 triệu.", 1),
    ("Thông báo khóa SIM hai chiều sau 2 giờ nữa do chưa chuẩn hóa thông tin thuê bao, gọi ngay số 0912345678.", 1),
    ("Bạn có một bưu kiện chưa thanh toán phí hải quan 35.000đ, truy cập bit.ly/giaohang-vn để hoàn tất nhận hàng.", 1),
    ("Hệ thống phát hiện đăng nhập lạ trên tài khoản MB Bank, vui lòng cung cấp mã OTP gửi về điện thoại để hủy giao dịch.", 1),
    ("Hỗ trợ vay tiền online giải ngân trong 5 phút chỉ cần CMND, lãi suất 0% tháng đầu, bấm đăng ký ngay.", 1),
    ("Chúc mừng số điện thoại của bạn may mắn trúng xe máy SH 150i, đóng phí làm hồ sơ 2 triệu để nhận xe.", 1),
    ("Anh ơi em đang kẹt tiền gấp, chuyển giúp em 5 triệu vào số tài khoản 1903888999 tối em gửi lại ngay.", 1),
    ("Trung tâm hỗ trợ hoàn tiền mua hàng Shopee lên tới 80%, vào nhóm Zalo nhận nhiệm vụ đơn hàng ảo ngay hôm nay.", 1),
    ("Thông báo phạt nguội giao thông quá hạn, truy cập http://csgt-tracuuphat.com để nộp phạt tránh bị tước bằng lái.", 1),
    ("Cảnh báo tài khoản VNeID bị lỗi đồng bộ mức 2, truy cập ngay m-vneid.com để cập nhật dữ liệu dân cư.", 1),
    ("Sàn chứng khoán quốc tế cam kết lợi nhuận 30% mỗi tuần, nạp 10 triệu tặng ngay 5 triệu tiền thưởng.", 1),
    ("Bên em hỗ trợ rút tiền mặt thẻ tín dụng phí 0%, anh quét mã QR này rồi cung cấp số tài khoản để đẩy tiền về.", 1),
    ("em ben nhan su cty X, tuyen ctv xem clip tiktok kiem tien, moi luot like dc 20k, ko mat phi gi dau lam thu di", 1),
    ("Tài khoản Facebook của bạn vi phạm bản quyền và sẽ bị vô hiệu hóa sau 24h, kháng nghị tại http://fb-support-check.info", 1),
    ("Cơ hội đổi thưởng điểm tích lũy Viettel sắp hết hạn, nhấp vào link để đổi 500.000đ tiền mặt về ví.", 1),
    ("Bố mẹ ơi con bị tai nạn đang cấp cứu ở bệnh viện, chuyển gấp 20 triệu vào tài khoản bác sĩ này giúp con.", 1),
    ("Thông báo từ Tòa án nhân dân: Quý khách có giấy triệu tập khẩn cấp, bấm phím 9 hoặc liên hệ cán bộ điều tra.", 1),
    ("Tham gia dự án đào coin miễn phí trên điện thoại, rút tiền về ATM ngay trong ngày, đăng ký tại t.me/free_coin_vn", 1),
    ("Khoản vay tín chấp 50 triệu đã được phê duyệt, vui lòng chuyển trước 1.5 triệu tiền bảo hiểm khoản vay để giải ngân.", 1),
    ("Thẻ ngân hàng của quý khách sắp hết hạn sử dụng, cung cấp 16 số thẻ và mã CVV để nhân viên gia hạn miễn phí.", 1),

    # ── 25 English Scam ──
    ("URGENT: Your bank account has been suspended due to suspicious login. Click http://secure-bank-verify.xyz to restore access.", 1),
    ("Congratulations! You have won a $1,000 Amazon gift card. Claim your cash prize now at bit.ly/win-amz-now", 1),
    ("Elon Musk is giving away 5,000 BTC! Send 0.1 BTC to our verified address and receive 0.2 BTC back immediately.", 1),
    ("Netflix billing failed! Update your payment details within 24 hours to avoid account termination: http://netflx-pay.top", 1),
    ("Hi, we are hiring part-time remote reviewers. Earn $300-$800 daily just liking YouTube videos. Message me on WhatsApp!", 1),
    ("IRS Final Notice: You have an unclaimed tax refund of $2,450. Submit your SSN and bank details here to receive direct deposit.", 1),
    ("Apple Security Alert: Your Apple ID was locked. Verify your identity immediately at http://apple-id-unlock.club", 1),
    ("FREE USDT airdrop for first 500 users! Connect your MetaMask wallet seed phrase on http://airdrop-claim.io", 1),
    ("FedEx delivery failed: Wrong address. Pay a $2.99 redelivery fee at http://fedex-track-parcel.info to reschedule.", 1),
    ("PayPal warning: Unauthorized transfer of $899 detected. Reply with your 6-digit verification code to cancel.", 1),
    ("Guaranteed 10x crypto returns in 7 days! Join our VIP trading signal channel t.me/crypto_whale_10x", 1),
    ("You have been selected as our lucky lottery winner of £500,000! Email your full name and passport copy to claim.", 1),
    ("Microsoft Support: Your PC is infected with a Trojan virus. Call +1-800-555-0199 immediately before data is wiped.", 1),
    ("Binance notice: Withdrawals disabled due to KYC expiration. Re-verify passport at http://binance-kyc-fix.com", 1),
    ("Dear customer, your Coinbase account will be frozen in 12 hours. Log in now to confirm your recovery phrase.", 1),
    ("Make $5,000 a week working from home with zero experience! Click here to activate your starter kit today.", 1),
    ("HSBC Fraud Dept: Did you spend $1,200 at BestBuy? Reply NO and send your OTP code so we can block the card.", 1),
    ("Your parcel is waiting at customs. Click www.dhl-customs-pay.cc to pay the $4.50 clearance charge.", 1),
    ("Exclusive investment opportunity: Double your Ethereum in 48 hours with our AI arbitrage bot, 100% risk-free!", 1),
    ("Instagram Copyright Infringement: Your page will be deleted in 24h. Appeal form: http://ig-appeal-portal.net", 1),
    ("Congratulations! Your phone number won an iPhone 16 Pro Max. Just pay $5 shipping fee to claim your reward.", 1),
    ("We noticed illegal activity on your checking account. Transfer funds to our federal safe locker account immediately.", 1),
    ("Want free followers and verified blue badge? Enter your Instagram password at http://free-badge-login.xyz", 1),
    ("Chase Bank Alert: Your debit card is temporarily locked. Confirm your PIN and CVV at the link below.", 1),
    ("Get instant loan approval up to $25,000 with bad credit! Send $150 processing fee via gift card to release funds.", 1),

    # ── 25 Vietnamese Safe (including hard negatives) ──
    ("Hôm nay trời đẹp quá, chiều đi cà phê không mọi người ơi?", 0),
    ("Chiều nay mấy giờ mày đi học về thế, đợi tao với nhé.", 0),
    ("ShopeeFood giao đơn hàng số 432, vui lòng xuống sảnh nhận hàng.", 0),
    ("Em ơi check giùm anh báo cáo doanh thu tháng này nhé.", 0),
    ("Chúc mừng sinh nhật bạn nhé, chúc bạn tuổi mới thật nhiều sức khỏe và thành công!", 0),
    ("Gửi mình xin cái link Google Drive tài liệu họp chiều nay với.", 0),
    ("Nhờ bạn xác nhận giúp mình lịch họp giao ban lúc 14h chiều nay nhé, hơi gấp một chút.", 0),
    ("Ban giám đốc đang thảo luận kế hoạch đầu tư trang thiết bị cho quý 4 năm nay.", 0),
    ("Quý khách lưu ý: Ngân hàng không bao giờ yêu cầu cung cấp mã OTP hoặc mật khẩu qua điện thoại hay đường link lạ.", 0),
    ("Dạ hồ sơ mở thẻ tín dụng hôm trước của anh đã được ngân hàng duyệt rồi ạ.", 0),
    ("Cảm ơn anh đã thanh toán hóa đơn tiền điện tháng 10 thành công.", 0),
    ("Chiều nay 5 giờ đá bóng sân Mỹ Đình nhé, anh em nhớ mang giày đầy đủ.", 0),
    ("Mình đã chuyển tiền ăn trưa 120k qua Momo cho bạn rồi đó, kiểm tra giúp mình nha.", 0),
    ("Link bài giảng môn Học sâu tuần này thầy mới cập nhật trên LMS rồi các bạn vào xem nhé.", 0),
    ("Tối nay nhà mình ăn cơm ở ngoài hay về nấu ăn vậy mẹ?", 0),
    ("Chúc mừng đội tuyển Việt Nam đã giành chiến thắng thuyết phục tối qua!", 0),
    ("Báo cáo tài chính quý 3 cho thấy lãi suất huy động đang có xu hướng ổn định trở lại.", 0),
    ("Bạn bấm vào nút tải xuống trên trang chủ chính thức của trường để lấy thời khóa biểu nhé.", 0),
    ("Hôm nay tôi mệt quá, xin phép nghỉ làm buổi chiều để đi khám bệnh.", 0),
    ("Cuối tuần này nhóm mình đi dã ngoại ở Ba Vì, ai đi thì xác nhận trong nhóm chat trước thứ 6 nhé.", 0),
    ("Mã xác thực OTP cho giao dịch mua vé xem phim của bạn là 482910. Tuyệt đối không chia sẻ mã này cho bất kỳ ai.", 0),
    ("Anh gửi em file slide thuyết trình dự án AI Scam Shield để em xem trước khi bảo vệ nhé.", 0),
    ("Sáng mai có lịch bảo trì hệ thống điện từ 8h đến 10h, mọi người lưu ý sạc đầy laptop.", 0),
    ("Quán trà sữa đầu ngõ đang có chương trình mua 1 tặng 1 cho sinh viên vào thứ 4 hàng tuần.", 0),
    ("Chào buổi sáng cả nhà, chúc mọi người một tuần mới làm việc thật hiệu quả!", 0),

    # ── 25 English Safe (including hard negatives) ──
    ("Hey, are we still meeting for lunch at 12:30 today?", 0),
    ("Please find attached the Q3 financial summary for your review before tomorrow's board meeting.", 0),
    ("Happy birthday! Wishing you an amazing year ahead filled with happiness.", 0),
    ("Can you share the Google Docs link to the project specification so I can leave comments?", 0),
    ("Feel free to click the GitHub repository link below to check out our open-source release.", 0),
    ("Congratulations to our engineering team for shipping version 2.0 ahead of schedule!", 0),
    ("Reminder: Your dentist appointment is scheduled for Friday at 3:00 PM.", 0),
    ("Security tip: Never share your OTP verification code or password with anyone, including support staff.", 0),
    ("We are looking for feedback on our new coffee blend. Let us know what you think!", 0),
    ("The server maintenance window is complete and all production APIs are back online.", 0),
    ("Thanks for the urgent fix on the login bug! The build passed all unit tests.", 0),
    ("Our company is reviewing long-term investment strategies in renewable energy.", 0),
    ("Your Amazon package has been delivered to your front porch at 2:15 PM.", 0),
    ("Don't forget to verify your pull request checks before merging into the main branch.", 0),
    ("What time does the movie start tonight? I can pick up the tickets on my way.", 0),
    ("I transferred my share of the dinner bill to you via Venmo, thanks for covering!", 0),
    ("Here is the crypto market recap article we discussed during the economics seminar.", 0),
    ("Could you confirm if you will attend the workshop next Tuesday?", 0),
    ("Great job on the presentation today! The demo worked smoothly without any errors.", 0),
    ("Flight UA412 from San Francisco to Seattle is boarding at Gate B12.", 0),
    ("My laptop battery died during the call, give me five minutes to plug in and rejoin.", 0),
    ("The library book you reserved is now available for pickup at the front desk.", 0),
    ("Please update your local dependencies by running npm install before starting the dev server.", 0),
    ("We offer free Wi-Fi for all guests staying at the hotel conference center.", 0),
    ("See you at the gym at 6 PM! Don't forget your water bottle.", 0),
]

assert len(TEST_100_SAMPLES) == 100

eval_model.eval()
sample_texts = [normalize_text(t, has_url_hint=None) for t, _ in TEST_100_SAMPLES]
sample_labels = [y for _, y in TEST_100_SAMPLES]

pred_probs = []
with torch.no_grad():
    for i in range(0, len(sample_texts), 32):
        enc = tokenizer(sample_texts[i:i+32], max_length=CONFIG["max_length"], padding=True, truncation=True, return_tensors="pt").to(device)
        out = eval_model(**enc)
        pred_probs.extend(torch.softmax(out.logits.float(), dim=-1)[:, 1].cpu().tolist())

pred_labels = [1 if p >= BEST_THRESHOLD else 0 for p in pred_probs]
correct_100 = sum(int(p == y) for p, y in zip(pred_labels, sample_labels))
acc_100 = correct_100 / 100.0
f1_100 = f1_score(sample_labels, pred_labels)

print(f"=== 100-SAMPLE REALISTIC BENCHMARK ===")
print(f"Correct: {correct_100}/100 ({acc_100*100:.1f}%) | F1: {f1_100:.4f} | Threshold: {BEST_THRESHOLD:.4f}")
print(f"VI Scam: {sum(p==y for p,y in zip(pred_labels[:25], sample_labels[:25]))}/25")
print(f"EN Scam: {sum(p==y for p,y in zip(pred_labels[25:50], sample_labels[25:50]))}/25")
print(f"VI Safe: {sum(p==y for p,y in zip(pred_labels[50:75], sample_labels[50:75]))}/25")
print(f"EN Safe: {sum(p==y for p,y in zip(pred_labels[75:100], sample_labels[75:100]))}/25")

errors_100 = []
for idx, ((raw_txt, y), p_lbl, p_prob) in enumerate(zip(TEST_100_SAMPLES, pred_labels, pred_probs)):
    if p_lbl != y:
        errors_100.append({"idx": idx, "actual": y, "pred": p_lbl, "scam_prob": round(p_prob, 4), "text": raw_txt})

print(f"\nMisclassified samples ({len(errors_100)}):")
for e in errors_100:
    print(" ", e)

In [ ]:
eval_model_cpu = eval_model.cpu().half()
eval_model_cpu.save_pretrained(CONFIG["output_dir"], safe_serialization=True)
tokenizer.save_pretrained(CONFIG["output_dir"])

model_file_bytes = sum(f.stat().st_size for f in Path(CONFIG["output_dir"]).glob("*.safetensors"))
model_size_mb = round(model_file_bytes / (1024 * 1024), 2)

perf_report = {
    "model_name": CONFIG["model_name"],
    "threshold": round(BEST_THRESHOLD, 4),
    "max_length": CONFIG["max_length"],
    "dataset_stats": stats,
    "metrics": {
        "accuracy": round(xlmr_test_metrics["accuracy"] * 100, 2),
        "precision": round(xlmr_test_metrics["precision"] * 100, 2),
        "recall": round(xlmr_test_metrics["recall"] * 100, 2),
        "f1_score": round(xlmr_test_metrics["f1"] * 100, 2),
        "roc_auc": round(xlmr_test_metrics["roc_auc"] * 100, 2),
        "pr_auc": round(xlmr_test_metrics["pr_auc"] * 100, 2),
    },
    "vi_bench_metrics": xlmr_vi_metrics,
    "vi_sms_metrics": xlmr_visms_metrics,
    "baseline_metrics": baseline_test_metrics,
    "benchmark_100": {
        "correct": correct_100,
        "total": 100,
        "f1": round(f1_100, 4),
        "errors": errors_100,
    },
    "system": {
        "latency_per_sample_ms": round(latency_ms, 2),
        "model_size_mb": model_size_mb,
    },
}

with open(os.path.join(CONFIG["artifacts_dir"], "system_performance_report.json"), "w", encoding="utf-8") as f:
    json.dump(perf_report, f, indent=2, ensure_ascii=False)

with open(os.path.join(CONFIG["output_dir"], "system_performance_report.json"), "w", encoding="utf-8") as f:
    json.dump(perf_report, f, indent=2, ensure_ascii=False)

zip_path = "/kaggle/working/exported_model.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in Path(CONFIG["output_dir"]).rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(CONFIG["output_dir"]))

print("Saved exported_model.zip:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")
print(json.dumps(perf_report["metrics"], indent=2))